In [478]:
import pandas as pd
import numpy as np
from datetime import datetime

n = 5000
today = datetime.today()

seed = np.random.randint(1, 65555)
rng = np.random.default_rng(seed)

In [479]:
df = pd.DataFrame()
df['gender'] = rng.choice(['Male', 'Female'], n, p=[0.58, 0.42])
df['age'] = (rng.gamma(3.0, 5.0, n) + 14).astype(int).clip(14, 85)

In [480]:
visit_probs = [
    0.10,  # 0
    0.35,  # 1
    0.35,  # 2
    0.15,  # 3
    0.03,  # 4
    0.015, # 5
    0.004, # 6
    0.001  # 7
]

visits_week = rng.choice(np.arange(8), size=n, p=visit_probs)

df['visits_per_week'] = visits_week.round(1)

def generate_weekly_visits(avg_visits, weeks=8, max_visits=7):
    visits = rng.poisson(lam=avg_visits, size=weeks)
    visits = np.clip(visits, 0, max_visits)
    return visits.astype(int)

weekly_visits_array = np.array([generate_weekly_visits(v) for v in df['visits_per_week']])

df['visits_last_7d'] = weekly_visits_array[:, 0]
df['visits_last_4w'] = weekly_visits_array[:, :4].sum(axis=1)
df['visits_prev_4w'] = weekly_visits_array[:, 4:8].sum(axis=1)

days_since_last_visit = np.empty(n, dtype=int)

mask_recent7 = df['visits_last_7d'] > 0
mask_30 = (df['visits_last_7d'] == 0) & (df['visits_last_4w'] > 0)
mask_prev30 = (df['visits_last_4w'] == 0) & (df['visits_prev_4w'] > 0)
mask_else = ~(mask_recent7 | mask_30 | mask_prev30)

def sample_weighted_days(rng, counts, low, high, base_lambda, factor, cap=10):
    chosen = np.empty(len(counts), dtype=int)
    days = np.arange(low, high + 1)
    for i, cnt in enumerate(counts):
        lam = base_lambda + factor * min(int(cnt), cap)
        weights = np.exp(-lam * (days - low))
        probs = weights / weights.sum()
        chosen[i] = rng.choice(days, p=probs)
    return chosen

idx = np.flatnonzero(mask_recent7)
if idx.size:
    counts = df.loc[idx, 'visits_last_7d'].to_numpy()
    days_since_last_visit[idx] = sample_weighted_days(
        rng, counts, low=0, high=6, base_lambda=0.6, factor=0.18, cap=7
    )

idx = np.flatnonzero(mask_30)
if idx.size:
    counts = df.loc[idx, 'visits_last_4w'].to_numpy()
    days_since_last_visit[idx] = sample_weighted_days(
        rng, counts, low=7, high=29, base_lambda=0.08, factor=0.02, cap=20
    )

idx = np.flatnonzero(mask_prev30)
if idx.size:
    counts = df.loc[idx, 'visits_prev_4w'].to_numpy()
    days_since_last_visit[idx] = sample_weighted_days(
        rng, counts, low=30, high=59, base_lambda=0.06, factor=0.01, cap=20
    )

idx = np.flatnonzero(mask_else)
if idx.size:
    counts = np.zeros(idx.size, dtype=int)
    days_since_last_visit[idx] = sample_weighted_days(
        rng, counts, low=60, high=365, base_lambda=0.008, factor=0.0, cap=1
    )

df['days_since_last_visit'] = days_since_last_visit

In [481]:
membership_config = {
    'Student':  {'price': 1990, 'duration': [30, 90]},
    'Basic':    {'price': 2990, 'duration': [30, 90, 180]},
    'Standard': {'price': 5490, 'duration': [90, 180, 365]},
    'Premium':  {'price': 9990, 'duration': [180, 365, 730]}
}

def assign_membership(age):
    types = list(membership_config.keys())
    if age < 23: return rng.choice(types, p=[0.7, 0.15, 0.12, 0.03])
    if age > 30: return rng.choice(types, p=[0.05, 0.2, 0.55, 0.2])
    return rng.choice(types, p=[0.1, 0.3, 0.45, 0.15])

df['membership_type'] = df['age'].apply(assign_membership)
df['membership_price'] = df['membership_type'].map(
    {k: v['price'] for k, v in membership_config.items()}
)

def assign_duration(membership_type):
    durations = membership_config[membership_type]['duration']

    weights = np.exp(-0.02 * np.array(durations))
    probs = weights / weights.sum()

    return rng.choice(durations, p=probs)

df['membership_duration_days'] = df['membership_type'].apply(assign_duration)

membership_end_date = []
last_visit_date = today - pd.to_timedelta(df['days_since_last_visit'], unit='D')

for i in range(n):
    duration = df.loc[i, 'membership_duration_days']
    last_visit = last_visit_date.iloc[i]

    min_end = last_visit
    max_end = today + pd.Timedelta(days=duration)

    random_days = rng.integers(0, (max_end - min_end).days + 1)
    membership_end_date.append(min_end + pd.Timedelta(days=random_days))

membership_end_date = pd.Series(membership_end_date)

df['membership_start_date'] = (
    membership_end_date - pd.to_timedelta(df['membership_duration_days'], unit='D')
).dt.date

df['membership_end_date'] = membership_end_date.dt.date
df['membership_days_to_expire'] = (membership_end_date - today).dt.days

In [482]:
price_norm = (df['membership_price'] - df['membership_price'].min()) / (
    df['membership_price'].max() - df['membership_price'].min()
)

churn_probability = (
    0.30 * (df['days_since_last_visit'] / 60) +
    0.35 * (-df['membership_days_to_expire'].clip(upper=0) / 60) +
    0.20 * (1 - df['visits_last_4w'] / 12) +
    0.15 * (1 - price_norm)
).clip(0, 1)

df['churn'] = rng.binomial(1, churn_probability)
df.loc[df['membership_days_to_expire'] < -60, 'churn'] = 1
df.loc[
    (df['days_since_last_visit'] > 45) &
    (df['membership_days_to_expire'] < 30),
    'churn'
] = 1

df.drop(columns=[
    'membership_type',
    'membership_start_date',
    'membership_end_date'
], inplace=True)

df.to_csv("gym_membership_churn.csv", index=False)

print(df['churn'].value_counts(normalize=True))

churn
0    0.7356
1    0.2644
Name: proportion, dtype: float64
